# RAG Chatbot with S-Biomed-Roberta and Phi-3.5

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using biomedical sentence embeddings (`pritamdeka/S-Biomed-Roberta-snli-multinli-stsb`) and a local language model (`phi-3.5`) to answer domain-specific medical questions. The pipeline includes:

- Document loading and splitting  
- Embedding generation with `S-Biomed-Roberta`  
- Vector store creation using `Chroma`  
- Retrieval-based question answering with `phi-3.5`  

This setup is ideal for medical QA tasks where factual grounding and domain relevance are critical.  
The model runs locally via `transformers` and Hugging Face pipelines, without requiring an API key.

---

In [1]:
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate
!pip install -U langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:

In [2]:
import os
import glob
import shutil

from huggingface_hub import login

from langchain.schema import Document
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

### Project Directory Connection


In [3]:
!git clone https://github.com/a20190202/PLN_Medical_Flashcard.git

fatal: destination path 'PLN_Medical_Flashcard' already exists and is not an empty directory.


In [4]:
!ls

PLN_Medical_Flashcard  sample_data


In [5]:
%cd PLN_Medical_Flashcard/pln_model

/content/PLN_Medical_Flashcard/pln_model


In [6]:
!pwd

/content/PLN_Medical_Flashcard/pln_model


# Vectorstore Generation
---

In [7]:
def read_txt_files(folder_path):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
    )

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        "source": filename,
                        "chunk_id": i,
                        "total_chunks": len(chunks)
                    }
                )
                all_docs.append(doc)

    return all_docs

all_documents = read_txt_files("data/textbooks")

## Embeddings model  
### `pritamdeka/S-Biomed-Roberta-snli-multinli-stsb`

This SentenceTransformer model is a fine-tuned version of `allenai/biomed_roberta_base` trained on multiple natural language inference (NLI) and semantic similarity datasets, including:

- SNLI, MNLI, and STS-B

It is optimized for sentence-level semantic similarity tasks in the biomedical domain, making it well-suited for generating dense embeddings of medical questions, abstracts, or documents for use in retrieval pipelines.

- **Base model:** `allenai/biomed_roberta_base`  
- **Embedding dimensions:** `768`  
- **Use case:** Biomedical sentence embeddings for retrieval, similarity search, and clustering

Model link: [https://huggingface.co/pritamdeka/S-Biomed-Roberta-snli-multinli-stsb](https://huggingface.co/pritamdeka/S-Biomed-Roberta-snli-multinli-stsb)


In [8]:
EMBEDDINGS = "pritamdeka/S-Biomed-Roberta-snli-multinli-stsb"

In [9]:
embeddings_model = HuggingFaceEmbeddings(
        model_name=EMBEDDINGS,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
# Create Vector Store (Run Only Once)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings_model,
    persist_directory=f"./{EMBEDDINGS.replace('/','_')}"
)

### Save the vector store after creation

In [11]:
# Define original path (correct one where Chroma actually saved the files)
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"{vectorstore_dir}.zip"

# Create the ZIP file from the original directory
shutil.make_archive(vectorstore_dir, 'zip', vectorstore_dir)

# Move the ZIP to /content so it's visible in Colab file browser
!mv "{zip_path}" /content/

print(f"Vector store zipped and moved to /content/: {os.path.basename(zip_path)}")

Vector store zipped and moved to /content/: pritamdeka_S-Biomed-Roberta-snli-multinli-stsb.zip


### Load the saved vector store

In [12]:
# Unzip the saved vector store
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"/content/{EMBEDDINGS.replace('/', '_')}.zip"

# Unzip only if not already extracted
if not os.path.exists(vectorstore_dir):
    shutil.unpack_archive(zip_path, vectorstore_dir)
    print(f"✅ Unzipped vector store to: {vectorstore_dir}")
else:
    print(f"ℹ️ Directory already exists: {vectorstore_dir}")

# Load the vector store
vectorstore = Chroma(
    persist_directory=vectorstore_dir,
    embedding_function=embeddings_model
)

ℹ️ Directory already exists: ./pritamdeka_S-Biomed-Roberta-snli-multinli-stsb


# RAG
---

In [13]:
# OLlama download
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
# Launch the Ollama server locally
!ollama serve > /dev/null 2>&1 &
!sleep 10

In [15]:
!ollama pull phi3:latest
!pip install -U langchain-ollama

In [16]:
from langchain_ollama import OllamaLLM

In [17]:
llm = OllamaLLM(
    model="phi3:latest",
    temperature=0.2,
    system=""
)

## Personalized prompt:

In [18]:
prompt = ChatPromptTemplate.from_template(
"""
Medical Flashcard Generator Prompt

You will receive the name of a medical condition or disease. Your task is to create 5 comprehensive flashcards that systematically cover the essential aspects of the condition for medical education purposes.

Required Coverage Areas:
1. Definition & Pathophysiology - Core concept and underlying mechanisms
2. Etiology & Risk Factors - Causes and predisposing factors
3. Clinical Presentation - Signs, symptoms, and clinical manifestations
4. Diagnostic Approach - Key tests, criteria, and differential considerations
5. Management & Treatment - Therapeutic interventions and prognosis

Flashcard Requirements:
- Each flashcard must contain one focused question and one comprehensive answer
- Questions should be clinically relevant and test practical knowledge
- Answers should be precise, direct, and medically accurate
- Provide specific details (lab values, medication dosages, timeframes where applicable)
- Use medical terminology appropriately while maintaining clarity
- Give concise, focused responses without bullet points or lists
- Prioritize high-yield information commonly tested in medical examinations

Context Considerations:
{context}

Output Format:
Flashcard 1: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

Flashcard 2: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

[Continue for all 5 flashcards]

Quality Standards:
- Ensure medical accuracy and evidence-based content
- Use current clinical guidelines and best practices
- Include relevant mnemonics or memory aids where helpful
- Maintain consistency in terminology and formatting
- Focus on clinically actionable information

Medical Condition: {input}
"""
 )

## RAG Pipeline:

In [19]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Inference Test:

### __Diabetes:__

In [20]:
question = "Diabetes"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Diabetes Mellitus
Q: What is the pathophysiological mechanism underlying type 2 diabetes mellitus?
A: Type 2 diabetes mellitus results from a combination of insulin resistance and an inadesfficient compensatory insulin secretory response. Over time, pancreatic beta cells fail to produce enough insulin or the body's tissues become less sensitive to circulating insulin due to factors such as obesity, physical inactivity, genetic predisposition, and aging. This leads to chronically elevated blood glucose levels (hyperglycemia), which can cause damage to various organ systems if left untreated.

Flashcard 2: Etiology & Risk Factors of Diabetes Mellitus
Q: What are the primary risk factors for developing type 2 diabetes mellitus?
A: The main etiological factor for type 2 diabetes is insulin resistance, which can be exacerbated by obesity (particularly visceral adiposity), physical inactivity, and genetic predisposition. Other risk fact

### __Asthma:__

In [21]:
question = "Asthma"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Asthma
Q: What is asthma, and how does its pathophysiology contribute to the symptoms experienced by patients?
A: Asthma is a chronic inflammatory disorder characterized by recurrent episodes of wheezing, breathlessness, chest tightness, and cough. The underlying mechanism involves intermittent airway obstruction due to bronchial smooth muscle cell hypertrophy and hyperreactivity, coupled with increased mucus secretion in the lower airways. Inflammation plays a central role, often involving eosinophils, mast cells, macrophages, lymphocytes, neutrophils, and epithelial cells that contribute to these pathological changes through various inflammatory mediators like cytokines and leukotrienes.

Flashcard 2: Etiology & Risk Factors of Asthma
Q: What are the primary etiological factors contributing to asthma, and what risk factors increase an individual's likelihood of developing this condition?
A: The development of asthma is multifact

### __Cardiac Arrest:__

In [22]:
question = "Cardiac Arrest"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1 - Definition & Pathophysiology
Q: What is cardiac arrest, and what are the primary physiological changes that occur during this event?
A: Cardiac arrest occurs when the heart's electrical system malfunctions, ceasing its ability to pump blood effectively. This results in a cessation of circulatory flow throughout the body except for brief periods where spontaneous circulation may resume (called 'shocks'). The immediate physiological changes include loss of consciousness due to insufficient brain perfusion and, if not rapidly reversed, can lead to irreversible organ damage or death.

Flashcard 2 - Etiology & Risk Factors
Q: What are the common etiologies for cardiac arrest in adults?
A: Cardiac arrests often result from acute myocardial infarction, particularly when a portion of the heart muscle is deprived of blood supply due to an occluding thrombus. Other risk factors include coronary artery disease (CAD), which can lead to unstable angina or silent myocardial 

### __Gastritis:__

In [23]:
question = "Gastritis"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Acute Erosive Hemorrhagic Gastropathy (Gastritis)
Q: What is the pathophysiological process that leads to erosions and hemorrhages in acute erosive hemorrhagic gastritis?
A: In cases of acute erosive hemorrhagic gastritis, active inflammation characterized by a marked neutrophil infiltrate is observed within the stomach lining. This results from damage to the mucosa that leads to disruption and loss of integrity at or near the basement membrane where epithelial cells reside. The presence of these immune cells, along with edema and hyperemia (increased blood flow), indicates an active inflammatory response which can lead to erosions—surface-level damage that exposes underlying tissue layers without complete separation from the basement membrane itself. Concurrently, hemorrhages may occur as capillaries within the mucosa are damaged or compromised due to this ongoing inflammation and disruption of normal gastric architecture.

Flash

### __Stroke:__

In [24]:
question = "Stroke"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Stroke
Q: What is the definition of a stroke, and what are its two primary types?
A: A stroke occurs when blood flow to an area of the brain is interrupted or reduced, depriving brain tissue of oxygen and nutrients. This can result in cell death within minutes. There are two main types of strokes: ischemic, caused by a blockage (such as a clot), accounting for about 87% of all cases; and hemorrhagic, resulting from bleeding into or around the brain due to ruptured blood vessels.

Flashcard 2: Etiology & Risk Factors for Stroke
Q: What are common risk factors associated with an increased chance of stroke?
A: Major modifiable and non-modifiable risk factors include hypertension, atrial fibrillation (AFib), heart disease such as congestive heart failure or rheumatic heart disease; diabetes mellitus; smoking; high cholester0.967821354
moler levels and obesity; a sedentary lifestyle; age (risk increases with age); gender, particularly 